# Lesson 03 - pandas 選欄、篩列、排序

1. 練習最常見的 pandas 操作：
2. 選欄位、依條件篩選列、排序，以及建立新欄位。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR_CANDIDATES = [
    Path("../data/raw"),
    Path("data/raw"),
    Path("/content/Py_dataAna/code/data/raw"),
    Path("/content/code/data/raw"),
]

DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if (path / "orders.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "找不到 CSV 資料。請確認 code/data/raw/ 內有 orders.csv 等資料檔，"
        "在 Colab 可先上傳整個專案資料夾或掛載 Google Drive。"
    )

print("Using data folder:", DATA_DIR.resolve())

Using data folder: E:\py_20260620\data\raw


In [2]:
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")
sessions = pd.read_csv(DATA_DIR / "sessions.csv")
events = pd.read_csv(DATA_DIR / "events.csv")
ab_assignments = pd.read_csv(DATA_DIR / "ab_assignments.csv")

tables = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "sessions": sessions,
    "events": events,
    "ab_assignments": ab_assignments,
}
pd.DataFrame(
    [{"table": name, "rows": len(df), "columns": len(df.columns)} for name, df in tables.items()]
)

,table,rows,columns
0,customers,2500,5
1,products,60,3
2,orders,22000,5
3,order_items,39627,5
4,sessions,70000,7
5,events,232067,6
6,ab_assignments,2500,3


In [ ]:
for i,j in tables.items():
    print(f'資料表名稱:{i}')
    print(j.info())#資料表的完整資訊

資料表名稱:customers
<class 'pandas.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   customer_id          2500 non-null   int64
 1   signup_date          2500 non-null   str  
 2   acquisition_channel  2500 non-null   str  
 3   city                 2500 non-null   str  
 4   segment              2500 non-null   str  
dtypes: int64(1), str(4)
memory usage: 97.8 KB
None
資料表名稱:products
<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   product_id  60 non-null     int64
 1   category    60 non-null     str  
 2   unit_price  60 non-null     int64
dtypes: int64(2), str(1)
memory usage: 1.5 KB
None
資料表名稱:orders
<class 'pandas.DataFrame'>
RangeIndex: 22000 entries, 0 to 21999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtyp

## 選欄位

In [ ]:
# .loc[row , col] 以文字為主
#print(ab_assignments.loc[:,'customer_id'])
#print(ab_assignments.loc[:,'customer_id':'assign_date'])
print(ab_assignments.loc[:,['experiment_group','customer_id']])

     experiment_group  customer_id
0                   B            1
1                   B            2
2                   A            3
3                   A            4
4                   A            5
...               ...          ...
2495                B         2496
2496                A         2497
2497                B         2498
2498                B         2499
2499                B         2500

[2500 rows x 2 columns]


In [14]:
# .iloc[row , col] 以整數為主
#print(ab_assignments.iloc[100:105 , 1])
#print(ab_assignments.iloc[100:105 , 0:2])
print(ab_assignments.iloc[100:105 , [2,0]])

    assign_date  customer_id
100  2024-01-11          101
101  2024-01-08          102
102  2024-01-10          103
103  2024-01-07          104
104  2024-01-02          105


In [17]:
#直接指定式的多欄選取
orders[["order_id",  "status","customer_id", "order_date"]].head()#前5筆資料


,order_id,status,customer_id,order_date
0,1,completed,2323,2025-05-02
1,2,completed,117,2024-07-14
2,3,cancelled,160,2025-03-29
3,4,completed,1240,2024-07-08
4,5,completed,714,2024-08-12


In [18]:
orders[["order_id",  "status","customer_id", "order_date"]].tail()#後5筆資料

,order_id,status,customer_id,order_date
21995,21996,completed,2390,2025-08-06
21996,21997,completed,2467,2025-10-04
21997,21998,refunded,1072,2024-07-05
21998,21999,completed,1977,2025-03-31
21999,22000,completed,843,2024-09-06


## 條件篩選

In [19]:
completed_orders = orders.loc[orders["status"] == "completed"]
completed_orders.head()

,order_id,customer_id,order_date,status,payment_type
0,1,2323,2025-05-02,completed,wallet
1,2,117,2024-07-14,completed,wallet
3,4,1240,2024-07-08,completed,atm
4,5,714,2024-08-12,completed,atm
5,6,2005,2025-01-14,completed,card


In [ ]:
completed_orders = orders.loc[(orders["status"] == "completed") & (orders["payment_type"] == "atm")]
completed_orders.head()

,order_id,customer_id,order_date,status,payment_type
3,4,1240,2024-07-08,completed,atm
4,5,714,2024-08-12,completed,atm
7,8,671,2024-04-22,completed,atm
9,10,211,2024-04-04,completed,atm
10,11,2091,2025-09-01,completed,atm


## 排序與取前幾筆

In [24]:
type(completed_orders["order_date"].values)

pandas.arrays.StringArray

In [25]:
completed_orders.info()

<class 'pandas.DataFrame'>
Index: 5111 entries, 3 to 21989
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   order_id      5111 non-null   int64
 1   customer_id   5111 non-null   int64
 2   order_date    5111 non-null   str  
 3   status        5111 non-null   str  
 4   payment_type  5111 non-null   str  
dtypes: int64(2), str(3)
memory usage: 239.6 KB


In [26]:
recent_completed = completed_orders.sort_values("order_date", ascending=False)#大到小
recent_completed.head(10)

,order_id,customer_id,order_date,status,payment_type
10034,10035,948,2025-12-31,completed,atm
7632,7633,2404,2025-12-31,completed,atm
9513,9514,2421,2025-12-31,completed,atm
13850,13851,891,2025-12-31,completed,atm
3307,3308,593,2025-12-31,completed,atm
3936,3937,860,2025-12-31,completed,atm
13097,13098,1027,2025-12-31,completed,atm
19425,19426,2293,2025-12-31,completed,atm
9316,9317,87,2025-12-30,completed,atm
1009,1010,2007,2025-12-30,completed,atm


## 建立新欄位

In [28]:
order_items = order_items.copy()
order_items["line_revenue"] = (
    order_items["quantity"] * order_items["unit_price"] * (1 - order_items["discount_rate"])
)
#order_items[["order_id", "quantity", "unit_price", "discount_rate", "line_revenue"]].head()
order_items.head()

,order_id,product_id,quantity,unit_price,discount_rate,line_revenue
0,1,28,1,567,0.05,538.65
1,2,2,1,3850,0.05,"3,657.50"
2,2,18,1,778,0.10,700.20
3,3,19,1,2193,0.00,"2,193.00"
4,3,36,1,1580,0.15,"1,343.00"


## 小練習

修改 `status` 或排序欄位，觀察輸出資料如何變化。

### 文字轉日期

In [29]:
completed_orders["order_date2"] = pd.to_datetime(completed_orders["order_date"])  
completed_orders.info()    

<class 'pandas.DataFrame'>
Index: 5111 entries, 3 to 21989
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      5111 non-null   int64         
 1   customer_id   5111 non-null   int64         
 2   order_date    5111 non-null   str           
 3   status        5111 non-null   str           
 4   payment_type  5111 non-null   str           
 5   order_date2   5111 non-null   datetime64[us]
dtypes: datetime64[us](1), int64(2), str(3)
memory usage: 279.5 KB
